# EDL Uncertainty Evaluation

This notebook evaluates uncertainty quantification for Wi-Fi Human Activity Recognition (Wi-HAR) using three approaches: softmax, EDL and GEN.

### Evaluation Tasks
1. **Accuracy vs. Uncertainty**: How model accuracy changes as uncertain samples are filtered out.
2. **OOD Detection (ROC/AUC)**: How well each method separates in-distribution from out-of-distribution (OOD) data.

## 1. Setup & Imports

Import libraries and define global paths, constants, and device configuration.

In [ ]:
from collections.abc import Callable
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import scienceplots  # noqa: F401
import torch
from edl_losses.edl import edl_inference
from edl_losses.gen import gen_inference
from sklearn.metrics import auc, roc_curve

from csi_vae.jobs import JobSettings, dataset, fusion, vae
from csi_vae.jobs.job import init_rng, make_dataloader
from csi_vae.studies import get_best_model, read_studies

plt.style.use(["science", "no-latex", "bright"])
plt.rcParams.update({"figure.dpi": 300, "svg.fonttype": "none"})

settings = JobSettings()

OUT_DIR = Path("../out")
FUSION_LAUNCH_DIR = OUT_DIR / "fusion"
PLOTS_DIR = OUT_DIR / "plots"
EDL_DIR = OUT_DIR / "edl"
WEIGHTS_DIR = Path("../weights")
OOD_DATASETS = [
    Path("../") / "dataset" / f"{ds}.h5"
    for ds in ["S1b", "S1c", "S2a", "S2b", "S2c", "S4a", "S4b", "S4c", "S6a", "S6b", "S6c"]
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ACTIVITIES = ["Walk", "Run", "Jump", "Sit", "Empty", "Stand", "Waving", "Clap", "Lay down", "Wipe", "Squat", "Stretch"]

PLOTS_DIR.mkdir(exist_ok=True)

## 2. Load Best Wi-HAR Model

Load the best-performing fusion model found by the Optuna hyperparameter search.

In [ ]:
studies = read_studies(str(FUSION_LAUNCH_DIR))
wi_har_results = get_best_model(studies)
init_rng(wi_har_results.seed)

wi_har_results

In [ ]:
def make_gaussians() -> list[vae.SingleAntenna]:
    """Create a list of SingleAntenna models based on the best model's parameters.

    Returns:
        A list of SingleAntenna models, one for each antenna, initialized with the parameters from the best model.

    """
    return [
        vae.SingleAntenna(
            settings.window_size,
            settings.n_subcarriers,
            wi_har_results.n_gaussians,
            vae.CONV_SPECS[wi_har_results.params["conv_layers_spec"]],
        ).to(DEVICE)
        for _ in range(settings.n_antennas)
    ]


wi_har_gaussians = make_gaussians()
wi_har_model = fusion.Delayed(
    wi_har_gaussians,
    wi_har_results.n_gaussians,
    settings.n_activities,
    wi_har_results.params["n_fusion_layers"],
    wi_har_results.params["fusion_dropout"],
).to(DEVICE)

model_path = (
    WEIGHTS_DIR
    / "fusion"
    / f"l{wi_har_results.n_gaussians}"
    / f"t{wi_har_results.trial_number}"
    / f"s{wi_har_results.seed}"
    / "delayed_fusion.pt"
)
wi_har_model.load_state_dict(torch.load(model_path))

## 3. Load Datasets

Load the in-distribution train/val/test splits and define a helper to load OOD datasets.

In [ ]:
train_ds, val_ds, test_ds = dataset.load(
    dataset_path=Path("../") / settings.dataset_path,
    window_size=settings.window_size,
    n_activities=settings.n_activities,
    stride=settings.stride,
)
full_ds = torch.utils.data.ConcatDataset([train_ds, val_ds, test_ds])

batch_size = 2 ** wi_har_results.params["batch_size_exp"]
train_dl = make_dataloader(train_ds, batch_size=batch_size, shuffle=True, seed=wi_har_results.seed)
val_dl = make_dataloader(val_ds, batch_size=batch_size, shuffle=False, seed=wi_har_results.seed)
test_dl = make_dataloader(test_ds, batch_size=batch_size, shuffle=False, seed=wi_har_results.seed)


def get_ood_dl(dataset_path: Path) -> torch.utils.data.DataLoader:
    """Load the OOD dataset and return a DataLoader for it."""
    _, _, ood_test_ds = dataset.load(
        dataset_path=dataset_path,
        window_size=settings.window_size,
        n_activities=settings.n_activities,
        stride=settings.stride,
    )
    return make_dataloader(ood_test_ds, batch_size=batch_size, shuffle=False, seed=wi_har_results.seed)

## 4. Load EDL and GEN Models

Load the best EDL and GEN models found by their respective Optuna studies.

In [ ]:
edl_study = optuna.load_study(
    study_name="edl",
    storage=f"sqlite:///{EDL_DIR}/edl.sqlite",
)
edl_gaussians = make_gaussians()
edl_model = fusion.Delayed(
    edl_gaussians,
    wi_har_results.n_gaussians,
    settings.n_activities,
    wi_har_results.params["n_fusion_layers"],
    wi_har_results.params["fusion_dropout"],
)
edl_model.to(DEVICE)
edl_model.load_state_dict(
    torch.load(WEIGHTS_DIR / "edl" / "edl" / f"t{edl_study.best_trial.number}" / "delayed_fusion.pt"),
)

In [ ]:
gen_study = optuna.load_study(
    study_name="gen",
    storage=f"sqlite:///{EDL_DIR}/gen.sqlite",
)
gen_gaussians = make_gaussians()
gen_model = fusion.Delayed(
    gen_gaussians,
    wi_har_results.n_gaussians,
    settings.n_activities,
    wi_har_results.params["n_fusion_layers"],
    wi_har_results.params["fusion_dropout"],
).to(DEVICE)
gen_model.load_state_dict(
    torch.load(WEIGHTS_DIR / "edl" / "gen" / f"t{gen_study.best_trial.number}" / "delayed_fusion.pt"),
)

## 5. Accuracy vs. Uncertainty

Evaluate how model accuracy improves as highly uncertain samples are progressively filtered out. A well-calibrated uncertainty estimate should yield higher accuracy at lower uncertainty thresholds.

In [ ]:
def _softmax_inference(logits: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Inference function for softmax-based model."""
    logits = logits.float()
    probs = torch.nn.functional.softmax(logits, dim=1)
    pred = probs.argmax(dim=1)
    k = logits.shape[-1]
    eps = 1e-10
    entropy = -torch.sum(probs * torch.log(probs + eps), dim=-1)  # [0, log(K)]
    uncertainty = entropy / torch.log(torch.tensor(float(k)))  # normalized to [0, 1]

    return pred, uncertainty, probs

In [ ]:
def accuracy_vs_uncertainty(
    model: torch.nn.Module,
    inference_fn: Callable,
    dataloader: torch.utils.data.DataLoader,
    device: torch.device,
) -> tuple[np.ndarray, list[float]]:
    """Compute accuracy vs uncertainty curve for the given model and dataloader.

    Arguments:
        model (torch.nn.Module): The trained model to evaluate.
        inference_fn (Callable): A function that takes the model's logits and returns predictions and uncertainties.
        dataloader (torch.utils.data.DataLoader): DataLoader for the dataset to evaluate on.
        device (torch.device): The device to run the model on.

    Returns:
        tuple: A tuple containing:
            - np.ndarray: Array of uncertainty thresholds.
            - list[float]: List of accuracies corresponding to each uncertainty threshold.

    """
    model.eval()
    all_uncertainties = []
    all_correct = []

    with torch.no_grad():
        for x, y in dataloader:
            with torch.autocast(device_type=device.type, dtype=torch.float16):
                logits = model(x.to(device))

            # Using your edl_inference function
            pred, uncertainty, _ = inference_fn(logits.float())

            correct = (pred == y.to(device)).cpu().numpy()
            all_correct.extend(correct)
            all_uncertainties.extend(uncertainty.cpu().numpy())

    all_correct = np.array(all_correct)
    all_uncertainties = np.array(all_uncertainties)

    # Define thresholds (from 0.0 to 1.0)
    thresholds = np.linspace(0, 1, 100)
    accuracies = []

    for t in thresholds:
        mask = all_uncertainties <= t
        if np.any(mask):
            acc = np.mean(all_correct[mask])
            accuracies.append(acc)
        else:
            accuracies.append(None)

    return thresholds, accuracies


# Collect data
softmax_thresholds, softmax_accuracies = accuracy_vs_uncertainty(wi_har_model, _softmax_inference, test_dl, DEVICE)
edl_thresholds, edl_accuracies = accuracy_vs_uncertainty(edl_model, edl_inference, test_dl, DEVICE)
gen_thresholds, gen_accuracies = accuracy_vs_uncertainty(gen_model, gen_inference, test_dl, DEVICE)

plt.figure(figsize=(5, 4))

# Accuracy Curve
plt.plot(softmax_thresholds, softmax_accuracies, lw=2, label="Softmax")
plt.plot(edl_thresholds, edl_accuracies, lw=2, label="EDL")
plt.plot(gen_thresholds, gen_accuracies, lw=2, label="GEN")

plt.xlabel("Uncertainty Threshold ($u$)")
plt.ylabel("Accuracy")
plt.tick_params(axis="y")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "edl_thresholds.pdf")
plt.show()

## 6. OOD Detection via ROC Curve

Measure how well each model's uncertainty score separates in-distribution (ID) from out-of-distribution (OOD) samples using the AUC-ROC metric.

A sample with uncertainty exceeding what is expected on training data (modelled as a Beta distribution) is flagged as OOD.

In [ ]:
from scipy.stats import beta as beta_dist


def beta_interval(
    model: fusion.Delayed,
    inference_fn: Callable,
    train_dl: torch.utils.data.DataLoader,
    interval: float = 0.95,
) -> tuple[float, float, float, float]:
    """Compute a Beta distribution credible interval for in-distribution uncertainty.

    Fits a Beta distribution to the uncertainty scores on the training set, then
    returns the interval bounds at the requested confidence level along with the
    fitted shape parameters.

    Arguments:
        model: The trained fusion.Delayed model to evaluate.
        inference_fn: A function that takes the model's logits and returns
            (predictions, uncertainties, probabilities).
        train_dl: DataLoader for the training dataset used to fit the Beta distribution.
        interval: Confidence level for the credible interval (default: 0.95).

    Returns:
        A tuple (lower, upper, alpha, beta) where lower/upper are the interval
        bounds and alpha/beta are the fitted Beta distribution shape parameters.

    """
    model.eval().to(DEVICE)
    uncertainties = []

    with torch.no_grad():
        for x, y in train_dl:
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16):
                logits = model(x.to(DEVICE))
            pred, uncertainty, _ = inference_fn(logits.float())

            correct = (pred == y.to(DEVICE)).cpu().numpy()
            correct_uncertainties = uncertainty.cpu().numpy()[correct]
            uncertainties.append(correct_uncertainties)

    uncertainties = np.concatenate(uncertainties)
    uncertainties = np.clip(uncertainties, 1e-6, 1 - 1e-6)

    a, b, _, _ = beta_dist.fit(uncertainties, floc=0, fscale=1)
    lower, upper = beta_dist.interval(interval, a, b)
    return lower, upper, float(a), float(b)  # pyright: ignore[reportArgumentType]

In [ ]:
models = {
    "Softmax": (wi_har_model, _softmax_inference),
    "EDL": (edl_model, edl_inference),
    "GEN": (gen_model, gen_inference),
}

for name, (model, inference_fn) in models.items():
    lower, upper, a, b = beta_interval(model, inference_fn, train_dl)
    print(f"{name} Beta Interval: [{lower:.4f}, {upper:.4f}] with a={a:.2f}, b={b:.2f}")

In [ ]:
from sklearn.neighbors import KernelDensity


def kde_fit(
    model: fusion.Delayed,
    inference_fn: Callable,
    train_dl: torch.utils.data.DataLoader,
    interval: float = 0.95,
) -> tuple[KernelDensity, float]:
    """Fit a KDE on training uncertainty scores and return the KDE and the threshold.

    The KDE is used to score test samples: the negative log-density under the
    in-distribution KDE is used as the OOD score (low density = more OOD).
    The threshold corresponds to the `interval`-th percentile of training
    uncertainties, giving a single operating point on the ROC.

    Arguments:
        model: The trained fusion.Delayed model to evaluate.
        inference_fn: A function that takes the model's logits and returns
            (predictions, uncertainties, probabilities).
        train_dl: DataLoader for the training dataset used to fit the KDE.
        interval: Confidence level for the operating-point threshold (default: 0.95).

    Returns:
        A tuple (kde, threshold) where kde is the fitted KernelDensity and
        threshold is the empirical percentile at the given confidence level.

    """
    model.eval().to(DEVICE)
    uncertainties = []

    with torch.no_grad():
        for x, y in train_dl:
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16):
                logits = model(x.to(DEVICE))
            pred, uncertainty, _ = inference_fn(logits.float())
            correct = (pred == y.to(DEVICE)).cpu().numpy()
            correct_uncertainties = uncertainty.cpu().numpy()[correct]
            uncertainties.append(correct_uncertainties)

    uncertainties = np.concatenate(uncertainties)

    kde = KernelDensity(kernel="gaussian", bandwidth="scott")
    kde.fit(uncertainties[:, None])

    threshold = np.percentile(uncertainties, interval * 100)
    return kde, threshold

In [ ]:
def plot_ood_detection_roc(
    models: dict,
    train_dl: torch.utils.data.DataLoader,
    id_dl: torch.utils.data.DataLoader,
    ood_dls: list[torch.utils.data.DataLoader],
    ood_dataset_names: list[str],
    device: torch.device,
) -> None:
    n = len(ood_dls)
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
    axes = axes.flatten()

    for idx, (ood_dl, ood_dataset_name) in enumerate(zip(ood_dls, ood_dataset_names, strict=True)):
        ax = axes[idx]
        for name, (model, inference_fn) in models.items():
            model.eval()
            scores = []
            binary_labels = []
            #_, upper, a, b = beta_interval(model, inference_fn, train_dl)
            kde, kde_threshold = kde_fit(model, inference_fn, train_dl)

            with torch.no_grad():
                for x, _ in id_dl:
                    with torch.autocast(device_type=device.type, dtype=torch.float16):
                        logits = model(x.to(device))
                    _, uncertainty, _ = inference_fn(logits.float())
                    scores.extend(uncertainty.cpu().numpy())
                    #scores.extend((uncertainty.cpu().numpy() > upper).astype(int))
                    #scores.extend(beta_dist.cdf(uncertainty.cpu().numpy(), a, b))
                    scores.extend(-kde.score_samples(uncertainty.cpu().numpy()[:, None]))
                    binary_labels.extend([0] * len(x))

                for x, _ in ood_dl:
                    with torch.autocast(device_type=device.type, dtype=torch.float16):
                        logits = model(x.to(device))
                    _, uncertainty, _ = inference_fn(logits.float())
                    scores.extend(uncertainty.cpu().numpy())
                    #scores.extend((uncertainty.cpu().numpy() > upper).astype(int))
                    #scores.extend(beta_dist.cdf(uncertainty.cpu().numpy(), a, b))
                    scores.extend(-kde.score_samples(uncertainty.cpu().numpy()[:, None]))
                    binary_labels.extend([1] * len(x))

            fpr, tpr, _ = roc_curve(np.array(binary_labels), np.array(scores))
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc:.3f})")

        ax.plot([0, 1], [0, 1], color="k", linestyle="--", alpha=0.5)
        ax.set_title(ood_dataset_name)
        ax.set_xlabel("FPR")
        ax.set_ylabel("TPR")
        ax.legend(loc="lower right", fontsize=6)
        ax.grid(True, alpha=0.5)

    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle("Beta CDF")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "edl_roc_grid.pdf")
    plt.show()


plot_ood_detection_roc(
    models={
        "Softmax": (wi_har_model, _softmax_inference),
        "EDL": (edl_model, edl_inference),
        "GEN": (gen_model, gen_inference),
    },
    train_dl=train_dl,
    id_dl=test_dl,
    ood_dls=[get_ood_dl(ds) for ds in OOD_DATASETS],
    ood_dataset_names=[ds.stem for ds in OOD_DATASETS],
    device=DEVICE,
)